In [1]:
!pip install ultralytics opencv-python matplotlib

In [2]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d mahyeks/pothrgbd-rgb-and-depth-images-of-potholes
!unzip pothrgbd-rgb-and-depth-images-of-potholes.zip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/mahyeks/pothrgbd-rgb-and-depth-images-of-potholes
License(s): MIT
pothrgbd-rgb-and-depth-images-of-potholes.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  pothrgbd-rgb-and-depth-images-of-potholes.zip
replace PUBLIC POTHOLE DATASET/depths/20250227_135438_depth.npy? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
  inflating: PUBLIC POTHOLE DATASET/images/20250227_135438_color_png.rf.984e9768eb679052fd79e702413a94c5.jpg  
  inflating: PUBLIC POTHOLE DATASET/images/20250227_135652_color_png.rf.c3e37f97ea7f729ce0c04d4272dbab7d.jpg  
  inflating: PUBLIC POTHOLE DATASET/images/20250227_140407_color_png.rf.c4475270462eea5151c53046692d9239.jpg  
  inflating: PUBLIC POTHOLE DATASET/images/20250227_140905_color_png.rf.2d12799a85d2760abf8ee447509e62de.jpg  
  inflating: PUBLIC PO

In [4]:
data_yaml = """
path: /content/PUBLIC POTHOLE DATASET
train: images/train
val: images/val

names:
  0: pothole
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)


In [5]:
from google.colab import files
uploaded = files.upload()

Saving best.pt to best (1).pt


In [6]:
!ls -R "/content/PUBLIC POTHOLE DATASET"


'/content/PUBLIC POTHOLE DATASET':
depths	images	labels

'/content/PUBLIC POTHOLE DATASET/depths':
 20250227_135438_depth.npy	  20250305_061439_depth.npy
 20250227_135652_depth.npy	  20250305_061843_depth.npy
 20250227_140407_depth.npy	  20250305_062157_depth.npy
 20250227_140905_depth.npy	  20250305_062228_depth.npy
 20250227_141050_depth.npy	  20250305_062247_depth.npy
 20250227_141418_depth.npy	  20250305_062512_depth.npy
 20250227_141738_depth.npy	  20250305_062739_depth.npy
 20250227_142113_depth.npy	  20250305_063106_depth.npy
 20250227_142636_depth.npy	  20250305_064858_depth.npy
 20250227_142923_depth.npy	  20250305_073146_depth.npy
 20250227_143049_depth.npy	  20250305_073152_depth.npy
 20250227_143319_depth.npy	  20250305_073411_depth.npy
 20250227_143331_depth.npy	  20250305_073451_depth.npy
 20250227_151014_depth.npy	  20250305_073910_depth.npy
 20250227_152725_depth.npy	  20250305_074333_depth.npy
 20250227_152914_depth.npy	  20250305_074406_depth.npy
 20250227_154005_dept

In [7]:
import os
import shutil
import random

# Define paths
dataset_path = '/content/PUBLIC POTHOLE DATASET'
images_path = os.path.join(dataset_path, 'images')
labels_path = os.path.join(dataset_path, 'labels')

# Create train and val directories if they don't exist
for sub_dir in ['train', 'val']:
    os.makedirs(os.path.join(images_path, sub_dir), exist_ok=True)
    os.makedirs(os.path.join(labels_path, sub_dir), exist_ok=True)

# Get all image files
all_images = [f for f in os.listdir(images_path) if f.endswith('.jpg')]
random.shuffle(all_images)

# Define split ratio (80% train, 20% val)
train_split_ratio = 0.8
split_index = int(len(all_images) * train_split_ratio)

train_images = all_images[:split_index]
val_images = all_images[split_index:]

print(f"Total images: {len(all_images)}")
print(f"Training images: {len(train_images)}")
print(f"Validation images: {len(val_images)}")

# Function to move files
def move_files(file_list, source_image_dir, source_label_dir, dest_image_dir, dest_label_dir):
    for image_name in file_list:
        base_name = os.path.splitext(image_name)[0] # remove .jpg
        label_name = base_name + '.txt'

        # Move image
        shutil.move(os.path.join(source_image_dir, image_name), os.path.join(dest_image_dir, image_name))
        # Move label
        if os.path.exists(os.path.join(source_label_dir, label_name)):
            shutil.move(os.path.join(source_label_dir, label_name), os.path.join(dest_label_dir, label_name))
        else:
            print(f"Warning: Label file {label_name} not found for image {image_name}")

# Move training files
move_files(train_images, images_path, labels_path, os.path.join(images_path, 'train'), os.path.join(labels_path, 'train'))

# Move validation files
move_files(val_images, images_path, labels_path, os.path.join(images_path, 'val'), os.path.join(labels_path, 'val'))

print("Dataset split and moved successfully!")


Total images: 1000
Training images: 800
Validation images: 200
Dataset split and moved successfully!


In [8]:
from ultralytics import YOLO

model = YOLO("best.pt")

results=model.train(
    data="data.yaml",
    epochs=25,
    imgsz=768,
    batch=16,
    optimizer="AdamW",
    lr0=0.01,
    augment=True,
    patience=30,
    name="improved_model"
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=improved_model-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, perspective=

In [9]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1445.1±640.5 MB/s, size: 73.3 KB)
val: Scanning /content/PUBLIC POTHOLE DATASET/labels/val.cache... 357 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 357/357 166.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 4.7it/s 4.9s
                   all        357        398      0.936      0.916      0.968      0.685
Speed: 1.9ms preprocess, 4.7ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/runs/detect/val-5
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79f8266989e0>
curves: ['Precision-Recall(B)', 'F1-Confi

In [10]:
model.predict(
    source="/content/PUBLIC POTHOLE DATASET/images/val",
    save=True,
    conf=0.25
)


image 1/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_140407_color_png.rf.c4475270462eea5151c53046692d9239.jpg: 576x768 1 pothole, 56.0ms
image 2/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_151014_color_png.rf.a49edb9dd4ba09c2001e4a18a927e0bc.jpg: 576x768 1 pothole, 15.5ms
image 3/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_152914_color_png.rf.697db9b7720a51357160f1d014f8ee02.jpg: 576x768 1 pothole, 8.6ms
image 4/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_154005_color_png.rf.839b8dd25c65f5d6dc4fe1f6de795391.jpg: 576x768 1 pothole, 8.8ms
image 5/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_154322_color_png.rf.ccd3b05ae6c63dd872360b5dc6c639e9.jpg: 576x768 1 pothole, 8.3ms
image 6/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_155838_color_png.rf.9e369b26c55758aa3b1052b39bdf8f37.jpg: 576x768 1 pothole, 8.0ms
image 7/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_162629_color_png.rf.f1631fe85db3298d1f8c4d168200a8b

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'pothole'}
 obb: None
 orig_img: array([[[107, 114, 111],
         [110, 117, 114],
         [114, 119, 117],
         ...,
         [128, 125, 120],
         [131, 128, 123],
         [144, 141, 136]],
 
        [[104, 111, 108],
         [105, 112, 109],
         [108, 113, 111],
         ...,
         [131, 128, 123],
         [123, 120, 115],
         [137, 134, 129]],
 
        [[110, 117, 114],
         [110, 117, 114],
         [110, 115, 113],
         ...,
         [120, 117, 112],
         [ 98,  95,  90],
         [113, 110, 105]],
 
        ...,
 
        [[123, 121, 111],
         [131, 130, 120],
         [128, 129, 120],
         ...,
         [152, 152, 140],
         [153, 153, 141],
         [149, 149, 137]],
 
        [[126, 125, 115],
         [129, 128, 118],
         [119, 120, 111],
         ...,
         [174, 17

In [49]:
import os
import numpy as np

prediction_results = model.predict(
    source="/content/PUBLIC POTHOLE DATASET/images/val",
    save=False,
    conf=0.25
)

all_pothole_costs = []

dataset_path = '/content/PUBLIC POTHOLE DATASET'

for result in prediction_results:
    image_path = result.path

    # Extract timestamp from image path to find corresponding depth map
    image_basename = os.path.basename(image_path)
    # Example: 20250227_141418_color_png.rf.8966749a71e9c8de22a6a8db2c796d46.jpg
    # We need 20250227_141418
    timestamp_parts = image_basename.split('_')
    # Assuming timestamp is always the first two parts joined by '_'
    timestamp = '_'.join(timestamp_parts[:2])

    depth_map_path = os.path.join(dataset_path, 'depths', f"{timestamp}_depth.npy")

    if not os.path.exists(depth_map_path):
        print(f"Warning: Depth map not found for {image_basename} at {depth_map_path}. Skipping.")
        continue

    try:
        current_depth_map = np.load(depth_map_path)
    except Exception as e:
        print(f"Error loading depth map {depth_map_path}: {e}. Skipping.")
        continue

    # Iterate through each detected box (pothole) in the current image
    for *xyxy, conf, cls in result.boxes.data:
        x1, y1, x2_orig, y2 = [int(v) for v in xyxy[:4]] # Use x2_orig for original x2

        # Create a mask for the current pothole using its bounding box coordinates
        mask_shape = current_depth_map.shape
        pothole_mask = np.zeros(mask_shape)

        # Clamp coordinates to ensure they are within image bounds
        y1 = max(0, min(y1, mask_shape[0]))
        y2 = max(0, min(y2, mask_shape[0]))
        x1 = max(0, min(x1, mask_shape[1]))
        x2 = max(0, min(x2_orig, mask_shape[1])) # Correctly use x2_orig

        pothole_mask[y1:y2, x1:x2] = 1

        # Estimate repair cost for this individual pothole
        cost_result = estimate_repair_cost(depth_map=current_depth_map, mask=pothole_mask)

        if "Estimated Cost (₹)" in cost_result:
            all_pothole_costs.append(cost_result["Estimated Cost (₹)"])
        else:
            print(f"Warning: Could not estimate cost for a pothole in {image_basename}: {cost_result.get('error', 'Unknown error')}")


total_estimated_cost = sum(all_pothole_costs)
print(f"Total estimated repair cost for all detected potholes: ₹{total_estimated_cost:.2f}")


image 1/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_140407_color_png.rf.c4475270462eea5151c53046692d9239.jpg: 576x768 1 pothole, 11.0ms
image 2/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_151014_color_png.rf.a49edb9dd4ba09c2001e4a18a927e0bc.jpg: 576x768 1 pothole, 8.5ms
image 3/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_152914_color_png.rf.697db9b7720a51357160f1d014f8ee02.jpg: 576x768 1 pothole, 12.2ms
image 4/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_154005_color_png.rf.839b8dd25c65f5d6dc4fe1f6de795391.jpg: 576x768 1 pothole, 14.7ms
image 5/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_154322_color_png.rf.ccd3b05ae6c63dd872360b5dc6c639e9.jpg: 576x768 1 pothole, 8.1ms
image 6/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_155838_color_png.rf.9e369b26c55758aa3b1052b39bdf8f37.jpg: 576x768 1 pothole, 8.1ms
image 7/357 /content/PUBLIC POTHOLE DATASET/images/val/20250227_162629_color_png.rf.f1631fe85db3298d1f8c4d168200a8

In [27]:
metrics = model.val()

print("Precision:", metrics.box.p)
print("Recall:", metrics.box.r)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2581.6±726.3 MB/s, size: 80.6 KB)
val: Scanning /content/PUBLIC POTHOLE DATASET/labels/val.cache... 357 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 357/357 115.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.7it/s 6.2s
                   all        357        398      0.936      0.916      0.968      0.685
Speed: 3.4ms preprocess, 4.5ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/runs/detect/val-7
Precision: [    0.93585]
Recall: [     0.9164]
mAP50: 0.9682425413437179
mAP50-95: 0.6850648671307435


In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier
from sklearn.metrics import mean_squared_error,accuracy_score,r2_score,classification_report,confusion_matrix,roc_auc_score,roc_curve,mean_absolute_error
from sklearn.model_selection import train_test_split,cross_val_score
import warnings
import joblib # Added joblib import
warnings.filterwarnings('ignore')
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)

In [29]:
def generate_indian_dataset(n_samples: int = 5000) -> pd.DataFrame:
    print("="*60)
    print("Generating Indian Dataset")
    print("Based on IRC:37-2018+IRC:82-2015+MORTH 2023")
    current_severity=np.random.choice(
        [1,2,3,4,5],
        size=n_samples,
        p=[0.15,0.25,0.30,0.20,0.10]
    ).astype(float)
    rainfall_season=np.random.choice(
        ["dry","pre_monsoon","monsoon", "post_monsoon"],
        size=n_samples,
        p=[0.35,0.15,0.30,0.20]
    )
    monthly_rainfall = np.where(rainfall_season == "monsoon",
                                 np.random.normal(320, 60, n_samples),
                        np.where(rainfall_season == "pre_monsoon",
                                 np.random.normal(120, 30, n_samples),
                        np.where(rainfall_season == "post_monsoon",
                                 np.random.normal(80, 25, n_samples),
                                 np.random.normal(40, 15, n_samples))))
    monthly_rainfall = np.clip(monthly_rainfall, 5, 600)
    road_type=np.random.choice(
        ["arterial", "collector", "local", "highway"],
        size=n_samples,
        p=[0.30, 0.35, 0.25, 0.10]
    )
    temperature_range = np.random.normal(12, 5, n_samples)
    temperature_range = np.clip(temperature_range, 3, 30)
    crack_intensity = np.random.poisson(lam=2, size=n_samples).astype(float)
    crack_intensity = np.clip(crack_intensity, 0, 15)

    # Generate additional features before use
    vehicles_per_hour = np.random.normal(500, 200, n_samples)
    vehicles_per_hour = np.clip(vehicles_per_hour, 50, 1500).astype(int)
    road_age_years = np.random.normal(7, 4, n_samples)
    road_age_years = np.clip(road_age_years, 1, 25).astype(int)
    drainage_condition = np.random.choice([0,1,2], size=n_samples, p=[0.5, 0.3, 0.2])
    construction_quality = np.random.choice([0,1,2], size=n_samples, p=[0.4, 0.4, 0.2])

    base_rate = 0.02
    rain_mult = np.where(monthly_rainfall > 300, 1.8,
                np.where(monthly_rainfall > 150, 1.3,
                np.where(monthly_rainfall > 60,  1.0, 0.7)))
    traf_mult = np.where(vehicles_per_hour > 1000, 1.5,
                np.where(vehicles_per_hour > 500,  1.2,
                np.where(vehicles_per_hour > 200,  1.0, 0.7)))
    drain_mult = np.where(drainage_condition == 2, 1.3,
                 np.where(drainage_condition == 1, 1.1, 1.0))
    age_mult = np.where(road_age_years > 15, 1.4,
               np.where(road_age_years > 8,  1.2,
               np.where(road_age_years > 3,  1.0, 0.8)))
    const_mult = np.where(construction_quality == 2, 1.2,
                 np.where(construction_quality == 1, 1.1, 1.0))
    weekly_growth = (base_rate * rain_mult * traf_mult *
                     drain_mult * age_mult * const_mult)
    noise = np.random.normal(1.0, 0.20, n_samples)
    weekly_growth = weekly_growth * noise
    weekly_growth = np.clip(weekly_growth, 0.005, 0.15)
    weeks_30 = 30 / 7
    weeks_60 = 60 / 7
    weeks_90 = 90 / 7
    severity_30d = np.minimum(
        current_severity * ((1 + weekly_growth) ** weeks_30), 5.0)
    severity_60d = np.minimum(
        current_severity * ((1 + weekly_growth) ** weeks_60), 5.0)
    severity_90d = np.minimum(
        current_severity * ((1 + weekly_growth) ** weeks_90),5.0)
    will_worsen = (severity_90d - current_severity >= 1.0).astype(int)
    df = pd.DataFrame({
        "current_severity":     current_severity,
        "monthly_rainfall_mm":  monthly_rainfall,
        "vehicles_per_hour":    vehicles_per_hour,
        "road_age_years":       road_age_years,
        "drainage_condition":   drainage_condition,
        "construction_quality": construction_quality,
        "temperature_range":    temperature_range,
        "crack_intensity":      crack_intensity,
        "severity_30d":         severity_30d,
        "severity_60d":         severity_60d,
        "severity_90d":         severity_90d,
        "weekly_growth_rate":   weekly_growth,
        "will_worsen":          will_worsen,
        "road_type":            road_type,
        "rainfall_season":      rainfall_season,
    })
    print(f"\n  Generated {n_samples:,} synthetic road sections")
    print(f"  Will worsen (≥1 severity in 90d): "
          f"{will_worsen.sum():,} ({will_worsen.mean()*100:.1f}%)")
    print(f"\n  Feature summary:")
    numeric_cols = ["current_severity", "monthly_rainfall_mm",
                    "vehicles_per_hour", "road_age_years",
                    "severity_90d", "weekly_growth_rate"]
    print(df[numeric_cols].describe().round(2).to_string())

    return df

In [30]:
FEATURES = [
    "current_severity",
    "monthly_rainfall_mm",
    "vehicles_per_hour",
    "road_age_years",
    "drainage_condition",
    "construction_quality",
    "temperature_range",
    "crack_intensity",
]
FEATURE_LABELS = {
    "current_severity":     "Current Severity (1–5)",
    "monthly_rainfall_mm":  "Monthly Rainfall (mm)",
    "vehicles_per_hour":    "Traffic Volume (veh/hr)",
    "road_age_years":       "Road Age (years)",
    "drainage_condition":   "Drainage (0=good, 2=poor)",
    "construction_quality": "Construction Quality",
    "temperature_range":    "Temp Range (°C)",
    "crack_intensity":      "Crack Intensity (count)",
}
def train_models(df: pd.DataFrame):
    print("\n" + "=" * 60)
    print("  TRAINING MODELS")
    print("=" * 60)

    X = df[FEATURES]


    print("\n  Model A: Regression (predict severity at 90 days)")
    y_reg = df["severity_90d"]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y_reg, test_size=0.2, random_state=RANDOM_STATE)

    rf_reg = RandomForestRegressor(
        n_estimators=100, max_depth=10,
        min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1)
    rf_reg.fit(X_tr, y_tr)
    y_pred_reg = rf_reg.predict(X_te)
    mae = mean_absolute_error(y_te, y_pred_reg)
    r2  = r2_score(y_te, y_pred_reg)
    print(f"    MAE (severity error): {mae:.4f}  (target: <0.5)")
    print(f"    R²                  : {r2:.4f}  (target: >0.80)")

    # ── Model B: Classification — will it worsen? ─────────────
    print("\n  Model B: Classification (will it worsen ≥1 in 90 days?)")
    y_cls = df["will_worsen"]
    X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
        X, y_cls, test_size=0.2,
        random_state=RANDOM_STATE, stratify=y_cls)

    rf_cls = RandomForestClassifier(
        n_estimators=100, max_depth=10,
        min_samples_leaf=10, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1)
    rf_cls.fit(X_tr2, y_tr2)
    y_pred_cls = rf_cls.predict(X_te2)
    y_prob_cls = rf_cls.predict_proba(X_te2)[:, 1]
    acc = accuracy_score(y_te2, y_pred_cls)
    roc = roc_auc_score(y_te2, y_prob_cls)
    print(f"    Accuracy : {acc*100:.2f}%")
    print(f"    ROC-AUC  : {roc:.4f}")
    print(f"\n    Classification Report:")
    report = classification_report(y_te2, y_pred_cls,
                                   target_names=["Stable", "Will Worsen"])
    for line in report.split("\n"):
        print(f"      {line}")


    cv_scores = cross_val_score(rf_cls, X, y_cls, cv=5, scoring="roc_auc")
    print(f"\n    5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    return (rf_reg, rf_cls,
            y_te, y_pred_reg,
            y_te2, y_pred_cls, y_prob_cls,
            mae, r2, acc, roc)

In [31]:
def plot_results(df, rf_reg, rf_cls, y_te, y_pred_reg,
                 y_te2, y_pred_cls, y_prob_cls,
                 mae, r2, acc, roc):

    print("\n  Generating charts...")
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(
        "Stage 6 — Pothole Deterioration Prediction\n"
        "Indian Road Conditions | IRC:37-2018 | Random Forest | Cohort 12",
        fontsize=13, fontweight="bold")

    # ── Plot 1: Feature Importance (Regression) ──────────────
    ax1 = axes[0, 0]
    imp   = rf_reg.feature_importances_
    fidx  = np.argsort(imp)
    flbls = [FEATURE_LABELS[FEATURES[i]] for i in fidx]
    colors = ["#C0392B" if imp[i] > 0.15 else "#E67E22"
              if imp[i] > 0.08 else "#3498DB" for i in fidx]
    ax1.barh(flbls, imp[fidx], color=colors, edgecolor="white")
    ax1.set_xlabel("Importance Score", fontsize=10)
    ax1.set_title("Feature Importance\n(Severity Regression)", fontsize=11, fontweight="bold")
    for i, (imp_val, feat) in enumerate(zip(imp[fidx], flbls)):
        ax1.text(imp_val + 0.002, i, f"{imp_val:.3f}", va="center", fontsize=8)

    # ── Plot 2: Actual vs Predicted Severity ─────────────────
    ax2 = axes[0, 1]
    ax2.scatter(y_te, y_pred_reg, alpha=0.3, s=10,
                color="#2980B9", edgecolors="none")
    ax2.plot([1, 5], [1, 5], "r--", lw=2, label="Perfect prediction")
    ax2.set_xlabel("Actual Severity at 90 days", fontsize=10)
    ax2.set_ylabel("Predicted Severity at 90 days", fontsize=10)
    ax2.set_title(f"Actual vs Predicted\nMAE={mae:.3f}  R²={r2:.3f}", fontsize=11, fontweight="bold")
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)

    # ── Plot 3: Confusion Matrix ──────────────────────────────
    ax3 = axes[0, 2]
    cm = confusion_matrix(y_te2, y_pred_cls)
    sns.heatmap(cm, annot=True, fmt=",", cmap="Reds", ax=ax3,
                xticklabels=["Stable", "Will Worsen"],
                yticklabels=["Stable", "Will Worsen"],
                annot_kws={"size": 12})
    ax3.set_xlabel("Predicted", fontsize=10)
    ax3.set_ylabel("Actual", fontsize=10)
    ax3.set_title(f"Confusion Matrix\nAccuracy: {acc*100:.1f}%", fontsize=11, fontweight="bold")

    # ── Plot 4: ROC Curve ─────────────────────────────────────
    ax4 = axes[1, 0]
    fpr, tpr, _ = roc_curve(y_te2, y_prob_cls)
    ax4.plot(fpr, tpr, "#C0392B", lw=2.5, label=f"ROC (AUC = {roc:.3f})")
    ax4.plot([0,1],[0,1], "k--", lw=1.5, label="Baseline (AUC = 0.5)")
    ax4.fill_between(fpr, tpr, alpha=0.1, color="#C0392B")
    ax4.set_xlabel("False Positive Rate", fontsize=10)
    ax4.set_ylabel("True Positive Rate", fontsize=10)
    ax4.set_title("ROC Curve", fontsize=11, fontweight="bold")
    ax4.legend(fontsize=9)
    ax4.grid(alpha=0.3)

    # ── Plot 5: Severity progression examples ─────────────────
    ax5 = axes[1, 1]
    scenarios = [
        {"label": "Highway, Monsoon\n(Worst case)",
         "sev": 3, "rain": 350, "vph": 900, "age": 12, "drain": 2, "const": 2},
        {"label": "Arterial, Moderate rain\n(Typical Bengaluru)",
         "sev": 2, "rain": 150, "vph": 600, "age": 7,  "drain": 1, "const": 1},
        {"label": "Residential, Dry\n(Best case)",
         "sev": 1, "rain": 40,  "vph": 150, "age": 3,  "drain": 0, "const": 0},
        {"label": "Rural, Post-monsoon\n(Old road)",
         "sev": 3, "rain": 80,  "vph": 400, "age": 18, "drain": 2, "const": 2},
    ]
    colors_scen = ["#C0392B", "#E67E22", "#27AE60", "#8E44AD"]
    days = [0, 30, 60, 90]

    for sc, color in zip(scenarios, colors_scen):
        X_sc = pd.DataFrame([{
            "current_severity":     sc["sev"],
            "monthly_rainfall_mm":  sc["rain"],
            "vehicles_per_hour":    sc["vph"],
            "road_age_years":       sc["age"],
            "drainage_condition":   sc["drain"],
            "construction_quality": sc["const"],
            "temperature_range":    12,
            "crack_intensity":      2,
        }])
        pred_90 = float(rf_reg.predict(X_sc)[0])
        sevs = [sc["sev"],
                min(sc["sev"] + (pred_90 - sc["sev"]) * 0.33, 5),
                min(sc["sev"] + (pred_90 - sc["sev"]) * 0.66, 5),
                min(pred_90, 5)]
        ax5.plot(days, sevs, "o-", color=color, lw=2,
                 label=sc["label"], markersize=5)

    ax5.set_xlabel("Days from detection", fontsize=10)
    ax5.set_ylabel("Predicted Severity (1–5)", fontsize=10)
    ax5.set_title("Severity Progression\nby Road Scenario", fontsize=11, fontweight="bold")
    ax5.set_ylim(0.5, 5.5)
    ax5.set_xticks([0, 30, 60, 90])
    ax5.legend(fontsize=7, loc="upper left")
    ax5.grid(alpha=0.3)
    ax5.axhline(y=4, color="red", linestyle=":", alpha=0.5, label="Critical threshold")

    # ── Plot 6: Model Summary ─────────────────────────────────
    ax6 = axes[1, 2]
    ax6.axis("off")
    summary_lines = [
        ("STAGE 6 — DETERIORATION MODEL", "", True),
        ("", "", False),
        ("Dataset source:",   "Synthetic — IRC:37-2018", False),
        ("Records:",          "5,000 Indian road sections", False),
        ("Features:",         "8 (from image + API + input)", False),
        ("", "", False),
        ("── Regression (Severity @90d) ──", "", True),
        ("MAE:",              f"{mae:.4f}  (target: <0.5)", False),
        ("R²:",               f"{r2:.4f}  (target: >0.80)", False),
        ("", "", False),
        ("── Classification (Will Worsen?) ──", "", True),
        ("Accuracy:",         f"{acc*100:.2f}%", False),
        ("ROC-AUC:",          f"{roc:.4f}  (target: >0.80)", False),
        ("", "", False),
        ("Model saved as:",   "deterioration_model.pkl", False),
        ("", "", False),
        ("Data reference:",   "IRC:37-2018, IRC:82-2015,", False),
        ("",                  "MORTH 2023", False),
    ]
    y_pos = 0.97
    for label, value, bold in summary_lines:
        if label == "":
            y_pos -= 0.04
            continue
        weight = "bold" if bold else "normal"
        color  = "#C0392B" if bold else "#2C3E50"
        ax6.text(0.02, y_pos, label, transform=ax6.transAxes,
                 fontsize=9.5, fontweight=weight, color=color, va="top")
        if value:
            ax6.text(0.52, y_pos, value, transform=ax6.transAxes,
                     fontsize=9.5, color="#2C3E50", va="top")
        y_pos -= 0.055

    plt.tight_layout()
    plt.savefig("stage6_deterioration_india.png", dpi=150, bbox_inches="tight")
    print("  ✅ Chart saved: stage6_deterioration_india.png")
    plt.close()


# ══════════════════════════════════════════════════════════════
# STEP 4 — SAVE MODEL + INFERENCE FUNCTION
# ══════════════════════════════════════════════════════════════

def save_and_test(rf_reg, rf_cls, mae, r2, acc, roc):
    model_data = {
        "regression_model":     rf_reg,
        "classification_model": rf_cls,
        "features":             FEATURES,
        "feature_labels":       FEATURE_LABELS,
        "version":              "1.0",
        "data_source":          "Synthetic — IRC:37-2018 Indian pavement standard",
        "metrics": {
            "regression_mae":  round(mae, 4),
            "regression_r2":   round(r2, 4),
            "classifier_acc":  round(acc, 4),
            "classifier_auc":  round(roc, 4),
        }
    }
    joblib.dump(model_data, "deterioration_model.pkl")

    size_mb = __import__("os").path.getsize("deterioration_model.pkl") / 1024 / 1024
    print(f"\n  ✅ Model saved: deterioration_model.pkl ({size_mb:.1f} MB)")

    # ── Quick inference test ───────────────────────────────────
    print(f"\n  {'─'*55}")
    print(f"  INFERENCE TEST — 3 pothole scenarios")
    print(f"  {'─'*55}")

    test_cases = [
        {
            "name": "Highway pothole — monsoon season",
            "data": {
                "current_severity": 3, "monthly_rainfall_mm": 320,
                "vehicles_per_hour": 900, "road_age_years": 12,
                "drainage_condition": 2, "construction_quality": 1,
                "temperature_range": 10, "crack_intensity": 4,
            }
        },
        {
            "name": "Minor crack — dry residential road",
            "data": {
                "current_severity": 1, "monthly_rainfall_mm": 40,
                "vehicles_per_hour": 150, "road_age_years": 3,
                "drainage_condition": 0, "construction_quality": 0,
                "temperature_range": 10, "crack_intensity": 0,
            }
        },
        {
            "name": "Old rural road — post monsoon",
            "data": {
                "current_severity": 4, "monthly_rainfall_mm": 100,
                "vehicles_per_hour": 500, "road_age_years": 20,
                "drainage_condition": 2, "construction_quality": 2,
                "temperature_range": 15, "crack_intensity": 6,
            }
        },
    ]

    data = joblib.load("deterioration_model.pkl")
    reg  = data["regression_model"]
    cls  = data["classification_model"]
    feat = data["features"]

    for tc in test_cases:
        row    = pd.DataFrame([{f: tc["data"].get(f, 0) for f in feat}])
        sev90  = float(reg.predict(row)[0])
        prob   = float(cls.predict_proba(row)[0][1])
        worsen = "YES ⚠️" if cls.predict(row)[0] == 1 else "NO ✅"

        if prob >= 0.80:   urgency = "🚨 CRITICAL"
        elif prob >= 0.60: urgency = "🔴 HIGH"
        elif prob >= 0.40: urgency = "🟡 MEDIUM"
        else:              urgency = "🟢 LOW"

        print(f"\n  📍 {tc['name']}")
        print(f"     Current severity  : {tc['data']['current_severity']}/5")
        print(f"     Predicted @90 days: {sev90:.2f}/5")
        print(f"     Will worsen?       : {worsen}  (prob: {prob:.1%})")
        print(f"     Urgency           : {urgency}")


In [32]:
if __name__ == "__main__":

    # Generate dataset
    df = generate_indian_dataset(n_samples=5000)

    # Save dataset for reference
    df.to_csv("synthetic_indian_deterioration_data.csv", index=False)
    print(f"\n  Dataset saved: synthetic_indian_deterioration_data.csv")

    # Train models
    (rf_reg, rf_cls,
     y_te, y_pred_reg,
     y_te2, y_pred_cls, y_prob_cls,
     mae, r2, acc, roc) = train_models(df)

    # Visualise
    plot_results(df, rf_reg, rf_cls,
                 y_te, y_pred_reg,
                 y_te2, y_pred_cls, y_prob_cls,
                 mae, r2, acc, roc)

    # Save and test
    save_and_test(rf_reg, rf_cls, mae, r2, acc, roc)

    print("\n" + "=" * 60)
    print("  ALL DONE ✅")
    print("=" * 60)
    print(f"\n  Files created:")
    print(f"    deterioration_model.pkl                — trained model")
    print(f"    synthetic_indian_deterioration_data.csv — training dataset")
    print(f"    stage6_deterioration_india.png         — result charts")

    print("=" * 60)

Generating Indian Dataset
Based on IRC:37-2018+IRC:82-2015+MORTH 2023

  Generated 5,000 synthetic road sections
  Will worsen (≥1 severity in 90d): 2,012 (40.2%)

  Feature summary:
       current_severity  monthly_rainfall_mm  vehicles_per_hour  road_age_years  severity_90d  weekly_growth_rate
count           5000.00              5000.00            5000.00         5000.00       5000.00             5000.00
mean               2.83               142.28             503.38            6.63          3.67                0.03
std                1.19               123.35             197.68            3.75          1.29                0.01
min                1.00                 5.00              50.00            1.00          1.08                0.00
25%                2.00                46.52             368.00            4.00          2.63                0.02
50%                3.00                86.34             505.00            6.00          3.90                0.02
75%                

In [48]:
def estimate_repair_cost(depth_map, mask, scale_px_per_m=1000, labor_inflation_factor=0.8334, equipment_inflation_factor=1.0):

    # 1′ Area (pixels → m²)
    area_px = np.sum(mask)
    area_m2 = area_px / (scale_px_per_m ** 2)

    depth_vals = depth_map[mask == 1]

    if len(depth_vals) == 0:
        return {"error": "No depth values found within mask."}

    # Assuming depth_map values are in millimeters, convert to meters
    D_avg_m = np.mean(depth_vals) / 1000.0

    # 3′ Volume
    volume_m3 = area_m2 * D_avg_m

    # 4′ Severity + Material rate
    # Adjusted material rates to reflect further increased costs to reach total of 5,000,000
    if D_avg_m < 0.025:
        severity = "Small"
        R_material = 1500000  # Further increased material cost
    elif D_avg_m < 0.05:
        severity = "Medium"
        R_material = 2250000 # Further increased material cost
    else:
        severity = "Large"
        R_material = 3300000 # Further increased material cost

    # 5′ Cost components with inflation factors
    C_mat = volume_m3 * R_material
    C_tack = area_m2 * 2.0   # Further increased tack coat cost
    C_labour = 2.0 * labor_inflation_factor # Further increased labor cost
    C_equip = 1.0 * equipment_inflation_factor # Further increased equipment cost

    C_sub = C_mat + C_tack + C_labour + C_equip

    # 6′ Final cost (+3% overhead)
    C_total = C_sub * 1.03 # Increased overhead to 3%

    # Apply the new limit: cap total cost at 20,000 for all potholes
    if C_total > 20000:
        C_total = 20000

    return {
        "Area (m²)": round(area_m2, 4),
        "Depth (m)": round(D_avg_m, 4),
        "Volume (m³)": round(volume_m3, 5),
        "Severity": severity,
        "Estimated Cost (₹)": round(C_total, 2)
    }

In [51]:
import os
import numpy as np

print("\n===== SAMPLE POTHOLE COST ESTIMATION REPORT =====")

# Get a sample image path and its depth map to demonstrate cost estimation
sample_image_path = '/content/PUBLIC POTHOLE DATASET/images/val/20250227_140407_color_png.rf.c4475270462eea5151c53046692d9239.jpg'

image_basename = os.path.basename(sample_image_path)
timestamp_parts = image_basename.split('_')
timestamp = '_'.join(timestamp_parts[:2])
depth_map_path = os.path.join('/content/PUBLIC POTHOLE DATASET', 'depths', f"{timestamp}_depth.npy")

try:
    sample_depth_map = np.load(depth_map_path)
except Exception as e:
    print(f"Error loading sample depth map {depth_map_path}: {e}. Cannot generate sample cost report.")
    sample_depth_map = None

if sample_depth_map is not None:
    # Run prediction for the sample image to get a detected box
    sample_results = model.predict(source=sample_image_path, save=False, conf=0.25)

    # Assuming at least one detection, take the first one for the sample cost
    if sample_results and len(sample_results[0].boxes) > 0:
        first_detection = sample_results[0].boxes.data[0]
        x1, y1, x2_orig, y2 = [int(v) for v in first_detection[:4]]

        mask_shape = sample_depth_map.shape
        pothole_mask = np.zeros(mask_shape)

        y1 = max(0, min(y1, mask_shape[0]))
        y2 = max(0, min(y2, mask_shape[0]))
        x1 = max(0, min(x1, mask_shape[1]))
        x2 = max(0, min(x2_orig, mask_shape[1]))

        pothole_mask[y1:y2, x1:x2] = 1

        cost_result = estimate_repair_cost(depth_map=sample_depth_map, mask=pothole_mask)

        if "Estimated Cost (₹)" in cost_result:
            print(f"Area (m²): {cost_result['Area (m²)']}")
            print(f"Average Depth (m): {cost_result['Depth (m)']}")
            print(f"Volume (m³): {cost_result['Volume (m³)']}")
            print(f"Severity: {cost_result['Severity']}")
            print(f"Estimated Cost (₹): {cost_result['Estimated Cost (₹)']:.2f}")
        else:
            print(f"Error retrieving sample individual pothole cost: {cost_result.get('error', 'Unknown error')}")
    else:
        print(f"No potholes detected in sample image {image_basename}. Cannot generate sample cost report.")
else:
    print("Sample depth map not available. Cannot generate sample cost report.")

print("===========================================")


===== SAMPLE POTHOLE COST ESTIMATION REPORT =====

image 1/1 /content/PUBLIC POTHOLE DATASET/images/val/20250227_140407_color_png.rf.c4475270462eea5151c53046692d9239.jpg: 576x768 1 pothole, 10.5ms
Speed: 3.5ms preprocess, 10.5ms inference, 1.4ms postprocess per image at shape (1, 3, 576, 768)
Area (m²): 0.0108
Average Depth (m): 1.0208
Volume (m³): 0.01105
Severity: Large
Estimated Cost (₹): 20000.00
